# Question 1: Does the adaptation gap matter?

**Question.** When a manga or light novel gets an anime, does the time between the source starting and the anime airing relate to how well the anime is rated? Is it different for manga and light novels?

**Data.** `src/build_adaptation_dataset.py` collected every TV anime from 2005–2024 on MyAnimeList that:
- is adapted from a manga, web manga, 4-koma manga, or light novel,
- has at least 1,000 ratings,
- and is a first season, not a sequel.

It then matched each one to its source by title search. It only accepted a match when the titles matched closely, the type fit, and the source started first. `src/flag_remakes.py` marked remakes of older anime.

Cells marked **YOUR TURN** are for you to write. The hints name the functions you'll probably want.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load the data

`gap_years` is the number of years from the source's first chapter or volume to the anime's first episode.

In [2]:
matched = pd.read_csv("../data/adaptation.csv")
unmatched = pd.read_csv("../data/adaptation_unmatched.csv")

for table in (matched, unmatched):
    table["anime_start"] = pd.to_datetime(table["anime_start"], format="mixed")
    table["anime_year"] = table["anime_start"].dt.year

matched["source_start"] = pd.to_datetime(matched["source_start"], format="mixed")
matched["gap_years"] = (matched["anime_start"] - matched["source_start"]).dt.days / 365.25

print(f"{len(matched)} matched, {len(unmatched)} could not be matched to a source")
matched.head()

1111 matched, 197 could not be matched to a source


,anime_id,anime_title,anime_title_english,source,anime_start,anime_score,anime_raters,anime_members,episodes,genres,...,source_id,source_title,source_type,source_start,source_score,source_members,earlier_version,is_remake,anime_year,gap_years
0,364,Gallery Fake,NaN,manga,2005-01-09,7.44,4024,17435,37,Adult Cast; Mystery; Seinen; Visual Arts,...,17026,Gallery Fake,manga,1992-01-01,NaN,800,NaN,False,2005,13.023956
1,406,Buzzer Beater,NaN,web_manga,2005-02-05,6.55,7056,17095,13,Sci-Fi; Sports; Team Sports,...,1221,Buzzer Beater,manga,1996-05-01,6.15,3081,NaN,False,2005,8.766598
2,485,Damekko Doubutsu,NaN,4_koma_manga,2005-01-17,7.22,5691,11849,26,Anthropomorphic; Comedy,...,4828,Damekko Doubutsu,manga,2001-09-27,NaN,204,NaN,False,2005,3.307324
3,342,Starship Operators,NaN,light_novel,2005-01-05,7.00,5862,17768,13,Drama; Military; Sci-Fi; Space,...,42125,Starship Operators,light_novel,2001-03-10,NaN,144,NaN,False,2005,3.824778
4,1196,Masuda Kousuke Gekijou: Gag Manga Biyori,NaN,manga,2005-02-07,7.13,3815,10814,12,Comedy; Gag Humor; Shounen,...,8584,Masuda Kousuke Gekijou: Gag Manga Biyori,manga,1999-12-06,6.94,630,NaN,False,2005,5.174538


**What each column means**

| column | meaning |
|---|---|
| `anime_score` | MyAnimeList's average user score (1–10) |
| `anime_raters` | how many users scored it |
| `anime_members` | how many users added it to their list |
| `source` | what MyAnimeList says the anime is adapted from |
| `source_score`, `source_members` | the same numbers for the manga or light novel itself |
| `title_match` | how closely the anime and source titles matched (1.0 means identical) |
| `is_remake`, `earlier_version` | whether an earlier TV anime of the same story exists |

## 2. Clean the data

### YOUR TURN: answer each of these in a code cell, then a sentence

1. **Remakes.** How many rows have `is_remake == True`? Remove them. They're re-adaptations, not first adaptations.
2. **One remake slipped through.** Look at the 10 largest `gap_years` with `.sort_values(...)`. *Gegege no Kitarou (2018)* is a remake that wasn't flagged. Remove it by `anime_id`, and check whether any others look like remakes.
3. **Missing scores.** Do any rows have a missing `anime_score`? Use `.isna().sum()`.
4. **Source categories.** `manga`, `web_manga`, and `4_koma_manga` are all manga. Decide whether to keep them separate or combine them into one `"manga"` group, and say why. Hint: `.replace({...})` on the `source` column.
5. **Who couldn't be matched?** Compare the median `anime_raters` and mean `anime_score` of `matched` and `unmatched`. If unmatched shows are less popular, what does that mean for your conclusions?

In [3]:
# 1. Remakes

In [4]:
# 2. Largest gaps

In [5]:
# 3. Missing scores

In [6]:
# 4. Source categories

In [7]:
# 5. Matched vs. unmatched

## 3. Charts

### YOUR TURN: make each chart, then write one sentence on what it shows

1. **How long is the usual gap?** A histogram of `gap_years`, one color per source type. Hint: loop over `df.groupby("source")` and call `plt.hist(..., alpha=0.5, label=...)`.
2. **Gap vs. score.** A scatter plot of `gap_years` (x) vs. `anime_score` (y). Add a trend line with `np.polyfit(x, y, 1)`. Most gaps are under 10 years, so try `plt.xlim(0, 15)` too.
3. **Score by gap bucket.** Use `pd.cut(df["gap_years"], [0, 2, 4, 6, 10, 60])` to make buckets, then a box plot of score per bucket (`df.boxplot(column="anime_score", by="gap_bucket")`).
4. **Has adapting gotten faster?** Median `gap_years` by `anime_year`, one line per source type. Do light novels get adapted faster than they used to?

In [8]:
# Chart 1

In [9]:
# Chart 2

In [10]:
# Chart 3

In [11]:
# Chart 4

## 4. Model

A gap could look like it matters just because popular sources get adapted faster *and* score higher. A regression lets you check the gap's effect while holding source popularity constant.

### YOUR TURN

```python
import statsmodels.formula.api as smf

model = smf.ols(
    "anime_score ~ gap_years + C(source) + np.log(source_members) + anime_year",
    data=df,
).fit()
print(model.summary())
```

Then answer:
- What is the coefficient on `gap_years`? In plain English: "holding the source's popularity, type, and year constant, each extra year of waiting goes with a ___ change in score."
- Is it statistically significant (p < 0.05)? Is it *large* enough to matter? A 0.01 change on a 10-point scale is tiny even if it's significant.
- Run the model again without `np.log(source_members)`. Does the `gap_years` coefficient change a lot? If it does, source popularity was hiding inside the gap.

In [12]:
# Model

## 5. Write-up

### YOUR TURN: 4–6 sentences
- What did you find?
- What's the most surprising chart?
- What can't this data tell you? For example: rater bias, the shows that couldn't be matched, sources that never got adapted at all, and correlation vs. causation.